# Embedding Projection Analysis

How do embeddings project into model spaces: attention Q/K/V projections,
MLP activations, direct embedding-to-unembed circuit, subspace utilization,
and token similarity structure.

## Why This Matters

Embedding projection examines the structure of the model's token representation spaces. The embedding and unembedding matrices define the model's interface with language, and their geometry constrains what semantic relationships the model can represent.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.embedding_projection_analysis import (
    embedding_to_attention_projection, embedding_to_mlp_projection,
    embedding_unembed_circuit, embedding_subspace_utilization,
    token_embedding_similarity_structure,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Embedding-to-Attention Projection

How do embeddings project into Q/K/V?

In [ ]:
result = embedding_to_attention_projection(model, tokens, layer=0)
print(f"Mean Q norm: {result['mean_q_norm']:.4f}, Mean V norm: {result['mean_v_norm']:.4f}\n")
for h in result['per_head']:
    print(f"  Head {h['head']}: Q={h['q_projection_norm']:.4f}, "
          f"K={h['k_projection_norm']:.4f}, V={h['v_projection_norm']:.4f}, "
          f"QK_ratio={h['qk_ratio']:.3f}")

## Embedding-to-MLP Projection

How many MLP neurons do the embeddings activate?

In [ ]:
result = embedding_to_mlp_projection(model, tokens, layer=0)
print(f"Mean activation fraction: {result['mean_activation_fraction']:.2%}\n")
for p in result['per_position']:
    bar = '█' * int(p['activation_fraction'] * 30)
    print(f"  Pos {p['position']}: {p['n_activated']:3d} active "
          f"({p['activation_fraction']:.1%}), max={p['max_activation']:.4f} {bar}")

## Direct Embedding-Unembed Circuit

What does the model predict from embeddings alone?

In [ ]:
result = embedding_unembed_circuit(model, tokens)
print(f"Self-prediction rate: {result['self_prediction_rate']:.2%}\n")
for p in result['per_position']:
    self_tag = '← SELF' if p['predicts_self'] else ''
    print(f"  Pos {p['position']}: input={p['input_token']}, "
          f"predicted={p['predicted_token']}, conf={p['confidence']:.4f}, "
          f"input_prob={p['input_token_probability']:.4f} {self_tag}")

## Subspace Utilization

How much of the embedding space do these tokens use?

In [ ]:
result = embedding_subspace_utilization(model, tokens)
print(f"d_model: {result['d_model']}, effective rank: {result['effective_rank']:.2f}")
print(f"Utilization: {result['utilization']:.2%}")
print(f"Dims for 90%: {result['dims_for_90_pct']}, for 95%: {result['dims_for_95_pct']}")
print(f"Mean norm: {result['mean_norm']:.4f} (±{result['norm_std']:.4f})")

## Token Similarity Structure

Which tokens have similar embeddings?

In [ ]:
result = token_embedding_similarity_structure(model, tokens)
print(f"Content similarity: {result['mean_content_similarity']:.4f}")
print(f"Full similarity: {result['mean_full_similarity']:.4f}")
print(f"Position effect: {result['position_effect']:+.4f}")
if result['most_similar_pair']:
    p = result['most_similar_pair']
    print(f"\nMost similar: pos {p['pos_a']}↔{p['pos_b']} (cos={p['content_similarity']:.4f})")
if result['least_similar_pair']:
    p = result['least_similar_pair']
    print(f"Least similar: pos {p['pos_a']}↔{p['pos_b']} (cos={p['content_similarity']:.4f})")